# NELLWICK AI Film Studio — NELLWICK AI Film Studio — Kaggle worker

This notebook is an interactive batch worker, not an always-on API server. Start it when you want to process queued story-planning, storyboard shot-planning, SDXL reference-image, Kokoro dialogue/narration or Wan video-generation jobs. It checkpoints each uploaded shot to Google Drive and reports job progress to your Studio.

**Before running:** create a worker token in Studio → Settings; connect Google Drive in Studio → Settings for video-generation jobs (planning jobs do not need Drive); enable a GPU runtime; the run cell uses the NELLWICK GitHub repository. If it is private, the notebook will request a read-only GitHub token without saving it. Never save your token in the notebook or repository.

The local AI story and shot planner uses Qwen3-4B or the Qwen3-1.7B low-VRAM fallback. The first video adapter runs Wan 2.1 T2V 1.3B for short text-to-video shots. A separate SDXL Base 1.0 adapter generates still reference images for characters and locations. SDXL needs a live T4 benchmark before routine use; Kokoro 82M local English voice generation is connected to the production queue; image-to-video, music generation and audio mixing remain separate adapters. First model load downloads several GB and may take time.


## Kaggle setup

For **Kaggle**, turn Internet on and select GPU in Notebook Settings. Kaggle documents free P100 availability, but queues and usage limits can vary. For **Colab**, select Runtime → Change runtime type → GPU. Colab does not guarantee free GPU availability. Do not use this notebook as a media server, tunnelled backend, or deepfake service.

The worker downloads Qwen3 for local story and shot planning. It selects Qwen3-4B on GPUs with at least 15 GiB total VRAM and Qwen3-1.7B on GPUs with 8–15 GiB. This threshold is not a guarantee: free GPU availability, free VRAM, system RAM and runtime limits vary. Planning does not need Drive OAuth; video jobs do. Qwen is unloaded before Wan video generation.


In [ ]:
# Installs runtimes without replacing the CUDA-enabled PyTorch supplied by Kaggle/Colab.
%pip install -q -U "diffusers>=0.35,<0.40" "transformers>=4.51,<5" accelerate safetensors sentencepiece imageio-ffmpeg requests "Pillow>=10" "kokoro>=0.9.4,<1.0" "soundfile>=0.12"

# Kokoro uses espeak-ng for English phoneme fallback.
!apt-get -qq -y install espeak-ng


In [ ]:
import os, getpass, subprocess, sys, tempfile
NELLWICK_STUDIO_URL = input("Studio URL (for local testing use your deployed Vercel URL): ").strip().rstrip("/")
NELLWICK_WORKER_TOKEN = getpass.getpass("Paste the one-time worker token created in Studio Settings: ").strip()
NELLWICK_MAX_JOBS = input("Maximum jobs for this notebook session [1]: ").strip() or "1"
os.environ["NELLWICK_STUDIO_URL"] = NELLWICK_STUDIO_URL
os.environ["NELLWICK_WORKER_TOKEN"] = NELLWICK_WORKER_TOKEN
os.environ["NELLWICK_MAX_JOBS"] = NELLWICK_MAX_JOBS


In [ ]:
# Clone the real NELLWICK repository. No token is stored in this notebook.
REPOSITORY_URL = "https://github.com/Freelankarx/nellwick-ai-film-studio.git"
WORKSPACE = "/content/nellwick-ai-film-studio" if "google.colab" in sys.modules else "/kaggle/working/nellwick-ai-film-studio"

if not os.path.isdir(WORKSPACE):
    clone_env = os.environ.copy()
    clone_env["GIT_TERMINAL_PROMPT"] = "0"
    clone_command = ["git", "clone", "--depth", "1", REPOSITORY_URL, WORKSPACE]
    clone_result = subprocess.run(clone_command, env=clone_env, capture_output=True, text=True)

    # Public repositories clone without credentials. If yours is private, enter a
    # read-only GitHub token only when prompted. It is passed to Git through a
    # temporary askpass helper and is not saved in the repo or Git remote URL.
    if clone_result.returncode != 0:
        access_token = getpass.getpass("GitHub read-only token (only if the repository is private): ").strip()
        if not access_token:
            raise RuntimeError(
                "Could not clone the repository. Confirm it is public, or rerun and "
                "provide a read-only token for a private repository. Details: "
                + clone_result.stderr[-1000:]
            )
        askpass_path = os.path.join(tempfile.gettempdir(), f"nellwick-git-askpass-{os.getpid()}.sh")
        with open(askpass_path, "w", encoding="utf-8") as askpass_file:
            askpass_file.write(
                "#!/bin/sh\n"
                "case \"$1\" in\n"
                "  *Username*) printf '%s' \"$NELLWICK_GIT_USERNAME\" ;;\n"
                "  *) printf '%s' \"$NELLWICK_GIT_TOKEN\" ;;\n"
                "esac\n"
            )
        os.chmod(askpass_path, 0o700)
        try:
            private_env = clone_env.copy()
            private_env["GIT_ASKPASS"] = askpass_path
            private_env["NELLWICK_GIT_USERNAME"] = "x-access-token"
            private_env["NELLWICK_GIT_TOKEN"] = access_token
            clone_result = subprocess.run(clone_command, env=private_env, capture_output=True, text=True)
        finally:
            try:
                os.remove(askpass_path)
            except OSError:
                pass
            if "private_env" in locals():
                private_env.pop("NELLWICK_GIT_TOKEN", None)
                private_env.pop("NELLWICK_GIT_USERNAME", None)
                private_env.pop("GIT_ASKPASS", None)
                del private_env
            del access_token
        if clone_result.returncode != 0:
            raise RuntimeError("GitHub clone failed: " + clone_result.stderr[-1200:])

# Always refresh an existing checkout so a notebook session uses the latest merged worker.
sync_env = os.environ.copy()
sync_env["GIT_TERMINAL_PROMPT"] = "0"
sync_result = subprocess.run(
    ["git", "-C", WORKSPACE, "pull", "--ff-only", "origin", "main"],
    env=sync_env,
    capture_output=True,
    text=True,
)
if sync_result.returncode != 0:
    raise RuntimeError("Could not update the NELLWICK worker from main: " + sync_result.stderr[-1200:])
print(sync_result.stdout.strip() or "NELLWICK worker is already up to date.")

worker_file = os.path.join(WORKSPACE, "workers", "shared", "worker.py")
if not os.path.isfile(worker_file):
    raise FileNotFoundError("workers/shared/worker.py not found. Confirm the repository branch contains the worker files.")
subprocess.run([sys.executable, worker_file, "--max-jobs", NELLWICK_MAX_JOBS], check=True)
